# wavesst — Full Pipeline Demo

End-to-end walkthrough of every transform in the library on two crossing chirps.

**Pipeline:** Signal synthesis → CWT → SST → MSST → STFT → STFT-SST → Ridge extraction → Reconstruction

All plots are interactive (requires ipywidgets ≥ 8.0 and a Jupyter environment).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import wavesst
from wavesst.viz.interactive import (
    iplot_cwt, iplot_sst, iplot_ridges,
    iplot_stft, iplot_stft_sst, iplot_components,
)

%matplotlib widget
print(f'wavesst {wavesst.__version__}')

## 1 — Signal Synthesis

Two crossing chirps:
- **Linear chirp:** 20 → 100 Hz over 2 s
- **Quadratic chirp:** 120 → 30 Hz over 2 s

Both have unit amplitude; the signal is a simple sum.

In [ ]:
FS = 512.0
T  = 2.0
t  = np.arange(int(T * FS)) / FS
N  = len(t)

# Linear chirp: 20 -> 100 Hz
f_lin  = 20 + 80 * t / T
chirp1 = np.cos(2 * np.pi * np.cumsum(f_lin) / FS)

# Quadratic chirp: 120 -> 30 Hz
f_quad = 120 - 90 * (t / T) ** 2
chirp2 = np.cos(2 * np.pi * np.cumsum(f_quad) / FS)

x = (chirp1 + chirp2).astype(np.float32)

fig, ax = plt.subplots(figsize=(12, 2))
ax.plot(t, x, lw=0.5)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.set_title('Two Crossing Chirps')
plt.tight_layout()
plt.show()

## 2 — Config

In [ ]:
cfg = wavesst.Config(device='auto', dtype='complex64')
print(cfg)

## 3 — CWT

Morlet CWT with 32 voices per octave.

In [ ]:
cwt_result = wavesst.cwt(x, fs=FS, nv=32, cfg=cfg)
print(f'CWT shape: {cwt_result.W.shape}  freqs: {cwt_result.freqs[0]:.1f}–{cwt_result.freqs[-1]:.1f} Hz')
iplot_cwt(cwt_result)

## 4 — SST

Synchrosqueezed CWT — energy is reassigned to the estimated instantaneous frequency.

In [ ]:
sst_result = wavesst.sst(x, fs=FS, nv=32, gamma='auto', cfg=cfg)
print(f'SST shape: {sst_result.Tx.shape}')
iplot_sst(sst_result)

## 5 — MSST (3 iterations)

Multi-synchrosqueezing progressively sharpens the TF plane.

In [ ]:
msst_result = wavesst.msst(x, fs=FS, nv=32, n_iter=3, gamma='auto', cfg=cfg)
print(f'MSST shape: {msst_result.Tx.shape}  n_iter={msst_result.n_iter}')
iplot_sst(msst_result)

## 6 — Ridge Extraction

Dynamic-programming ridge tracker on the SST plane.

In [ ]:
ridges = wavesst.extract_ridges(sst_result, n=2, penalty=2.0)
for i, r in enumerate(ridges):
    print(f'Ridge {i+1}: median freq = {np.median(r.freq_path):.1f} Hz,  energy = {r.energy:.2e}')
iplot_ridges(sst_result, ridges)

## 7 — Reconstruction (CWT-SST)

Admissibility-constant inverse: `x_k(t) = (2/C_ψ) · Re[ Σ_{f ∈ band_k} T_x(f, t) ]`.

In [ ]:
components = wavesst.reconstruct(sst_result, ridges)
iplot_components(components, sst_result.times)

## 8 — STFT

In [ ]:
stft_result = wavesst.stft(x, fs=FS, nperseg=256, noverlap=240, cfg=cfg)
print(f'STFT shape: {stft_result.V.shape}')
iplot_stft(stft_result)

## 9 — STFT-SST

In [ ]:
stft_sst_result = wavesst.stft_sst(x, fs=FS, nperseg=256, noverlap=240, gamma='auto', cfg=cfg)
print(f'STFT-SST shape: {stft_sst_result.Tx.shape}')
iplot_stft_sst(stft_sst_result)

## 10 — Reconstruction (STFT-SST)

Overlap-add synthesis on the raw STFT V at the ridge bin.

In [ ]:
stft_ridges = wavesst.extract_ridges(stft_sst_result, n=2, penalty=2.0)
stft_comps  = wavesst.reconstruct(stft_sst_result, stft_ridges, fs=FS)
iplot_components(stft_comps, t)